# Cross-Sell Association Rules Baseline

Bu notebook sepetlerden ürün-ürün birlikteliklerini öğrenir ve cross-sell önerileri üretir. Model adı: `cross_sell_association_rules`.

In [ ]:
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Project root could not be found from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "gold"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
METRICS_DIR = PROJECT_ROOT / "artifacts" / "metrics"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "recommendation_outputs"

TRAIN_PATH = DATA_DIR / "cross_sell_train_baskets.parquet"
TEST_PATH = DATA_DIR / "cross_sell_test_baskets.parquet"
MODEL_PATH = MODEL_DIR / "cross_sell_association_rules_baseline.pkl"
RULES_PATH = OUTPUT_DIR / "cross_sell_association_rules_baseline.parquet"
METRICS_PATH = METRICS_DIR / "cross_sell_association_rules_baseline_metrics.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH, TEST_PATH

## Parameters

In [ ]:
MODEL_NAME = "cross_sell_association_rules"

# Az görülen ürünleri elemek modeli daha stabil ve hızlı yapar.
MIN_ITEM_SUPPORT = 20
MIN_PAIR_SUPPORT = 5

# Çok büyük sepetlerde kombinasyon sayısı patlar.
MAX_BASKET_SIZE_FOR_PAIRS = 30

# Her ürün için saklanacak en iyi cross-sell adayları.
TOP_N_RULES_PER_ITEM = 200

# Değerlendirme ve öneri uzunluğu.
RECOMMEND_K = 12

# Test çok büyükse hızlı deneme için sayı ver; tamamını kullanmak için None bırak.
EVAL_SAMPLE_SIZE = 100_000
RANDOM_STATE = 42

## Load Train/Test

In [ ]:
def normalize_articles(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, set)):
        return [str(article) for article in value if pd.notna(article)]
    return [str(value)]


def unique_preserve_order(values):
    seen = set()
    output = []
    for value in values:
        if value not in seen:
            seen.add(value)
            output.append(value)
    return output


train_baskets = pd.read_parquet(TRAIN_PATH)
test_baskets = pd.read_parquet(TEST_PATH)

for frame in [train_baskets, test_baskets]:
    frame["t_dat"] = pd.to_datetime(frame["t_dat"])
    frame["articles"] = frame["articles"].apply(normalize_articles)
    frame["basket_size"] = frame["articles"].apply(len).astype("int32")

train_baskets.shape, test_baskets.shape

## Train Association Rules

In [ ]:
train_items = train_baskets[["articles"]].explode("articles").rename(columns={"articles": "article_id"})
item_support = train_items["article_id"].value_counts().astype("int64")
frequent_items = set(item_support[item_support >= MIN_ITEM_SUPPORT].index)
popular_items = item_support.index.tolist()

len(frequent_items), len(popular_items)

In [ ]:
pair_counts = Counter()
used_basket_count = 0

for articles in train_baskets["articles"]:
    basket = sorted(set(article for article in articles if article in frequent_items))
    if len(basket) < 2 or len(basket) > MAX_BASKET_SIZE_FOR_PAIRS:
        continue
    used_basket_count += 1
    pair_counts.update(combinations(basket, 2))

len(pair_counts), used_basket_count

In [ ]:
def build_rules_dataframe(pair_counts, item_support, n_baskets, min_pair_support):
    rows = []
    for (left, right), pair_count in pair_counts.items():
        if pair_count < min_pair_support:
            continue

        left_count = int(item_support.get(left, 0))
        right_count = int(item_support.get(right, 0))
        if left_count == 0 or right_count == 0:
            continue

        for antecedent, consequent, antecedent_count, consequent_count in [
            (left, right, left_count, right_count),
            (right, left, right_count, left_count),
        ]:
            support = pair_count / n_baskets
            confidence = pair_count / antecedent_count
            consequent_support = consequent_count / n_baskets
            lift = confidence / consequent_support if consequent_support > 0 else 0
            jaccard = pair_count / (antecedent_count + consequent_count - pair_count)
            score = confidence * np.log1p(lift) * np.log1p(pair_count)

            rows.append(
                {
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "pair_count": int(pair_count),
                    "antecedent_count": antecedent_count,
                    "consequent_count": consequent_count,
                    "support": support,
                    "confidence": confidence,
                    "lift": lift,
                    "jaccard": jaccard,
                    "score": score,
                }
            )

    rules = pd.DataFrame(rows)
    if rules.empty:
        return rules

    rules = rules.sort_values(
        ["antecedent", "score", "confidence", "lift", "pair_count"],
        ascending=[True, False, False, False, False],
    )
    rules = rules.groupby("antecedent", as_index=False).head(TOP_N_RULES_PER_ITEM)
    return rules.reset_index(drop=True)


rules = build_rules_dataframe(
    pair_counts=pair_counts,
    item_support=item_support,
    n_baskets=len(train_baskets),
    min_pair_support=MIN_PAIR_SUPPORT,
)

rules.head(10)

In [ ]:
rules_by_item = defaultdict(list)

for row in rules.itertuples(index=False):
    rules_by_item[row.antecedent].append(
        {
            "consequent": row.consequent,
            "score": float(row.score),
            "confidence": float(row.confidence),
            "lift": float(row.lift),
            "support": float(row.support),
            "pair_count": int(row.pair_count),
        }
    )

len(rules_by_item)

## Recommendation Function

In [ ]:
def recommend_for_basket(articles, rules_by_item, popular_items, k=12):
    context = unique_preserve_order(normalize_articles(articles))
    seen = set(context)
    candidate_scores = defaultdict(float)

    for article in context:
        for rule in rules_by_item.get(article, []):
            candidate = rule["consequent"]
            if candidate not in seen:
                candidate_scores[candidate] += rule["score"]

    ranked = [
        article for article, _ in sorted(candidate_scores.items(), key=lambda item: item[1], reverse=True)
    ]

    for article in popular_items:
        if len(ranked) >= k:
            break
        if article not in seen and article not in candidate_scores:
            ranked.append(article)

    return ranked[:k]


example_basket = test_baskets.iloc[0]["articles"][:1]
recommend_for_basket(example_basket, rules_by_item, popular_items, k=RECOMMEND_K)

## Evaluate On Test Baskets

In [ ]:
def context_target_split(articles):
    articles = unique_preserve_order(normalize_articles(articles))
    if len(articles) < 2:
        return None, None
    split_idx = max(1, len(articles) // 2)
    context = articles[:split_idx]
    target = set(articles[split_idx:])
    return context, target


def evaluate_baskets(test_frame, rules_by_item, popular_items, k=12):
    precisions = []
    recalls = []
    hit_rates = []

    for articles in test_frame["articles"]:
        context, target = context_target_split(articles)
        if not context or not target:
            continue

        recommendations = recommend_for_basket(context, rules_by_item, popular_items, k=k)
        hits = len(set(recommendations) & target)

        precisions.append(hits / k)
        recalls.append(hits / len(target))
        hit_rates.append(1.0 if hits > 0 else 0.0)

    return {
        "evaluated_baskets": int(len(precisions)),
        f"precision_at_{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"recall_at_{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"hit_rate_at_{k}": float(np.mean(hit_rates)) if hit_rates else 0.0,
    }


if EVAL_SAMPLE_SIZE is not None and len(test_baskets) > EVAL_SAMPLE_SIZE:
    eval_baskets = test_baskets.sample(EVAL_SAMPLE_SIZE, random_state=RANDOM_STATE)
else:
    eval_baskets = test_baskets

metrics = evaluate_baskets(eval_baskets, rules_by_item, popular_items, k=RECOMMEND_K)
metrics

## Save Model And Outputs

In [ ]:
model_artifact = {
    "model_name": MODEL_NAME,
    "params": {
        "min_item_support": MIN_ITEM_SUPPORT,
        "min_pair_support": MIN_PAIR_SUPPORT,
        "max_basket_size_for_pairs": MAX_BASKET_SIZE_FOR_PAIRS,
        "top_n_rules_per_item": TOP_N_RULES_PER_ITEM,
        "recommend_k": RECOMMEND_K,
    },
    "rules_by_item": dict(rules_by_item),
    "popular_items": popular_items,
}

rules.to_parquet(RULES_PATH, index=False)

with MODEL_PATH.open("wb") as f:
    pickle.dump(model_artifact, f)

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved rules: {RULES_PATH}")
print(f"Saved model: {MODEL_PATH}")
print(f"Saved metrics: {METRICS_PATH}")